In [ ]:
!pip install --upgrade gdown
import gdown
folder_url = 'https://drive.google.com/drive/folders/1s5W0Nfz9NEN8gP14tge8GuouUDXs2Ssq'
gdown.download_folder(folder_url, quiet=False, use_cookies=False)

  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.2
    Uninstalling gdown-5.2.2:
      Successfully uninstalled gdown-5.2.2


Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed


Processing file 1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF full-shape.zip
Processing file 1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p partial.zip
Processing file 1vwFQbYRDQlyxOZk0N0Hlxi8O_bhHxgIM rotate.zip


Downloading...
From (original): https://drive.google.com/uc?id=1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF
From (redirected): https://drive.google.com/uc?id=1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF&confirm=t&uuid=e06d86bc-f299-465d-91ce-54e48fa5bb3c
To: /content/AffordanceNet/full-shape.zip
100%|██████████| 558M/558M [00:08<00:00, 67.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p
From (redirected): https://drive.google.com/uc?id=1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p&confirm=t&uuid=b6711306-9bd4-4e1e-8f31-30e405b75347
To: /content/AffordanceNet/partial.zip
100%|██████████| 2.19G/2.19G [00:40<00:00, 53.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1vwFQbYRDQlyxOZk0N0Hlxi8O_bhHxgIM
To: /content/AffordanceNet/rotate.zip
100%|██████████| 1.15M/1.15M [00:00<00:00, 10.9MB/s]
Download completed


['/content/AffordanceNet/full-shape.zip',
 '/content/AffordanceNet/partial.zip',
 '/content/AffordanceNet/rotate.zip']

In [ ]:
!unzip -o /content/AffordanceNet/full-shape.zip -d /content/AffordanceNet/
!unzip -o /content/AffordanceNet/partial.zip -d /content/AffordanceNet/
!unzip -o /content/AffordanceNet/rotate.zip -d /content/AffordanceNet/
!rm /content/AffordanceNet/*.zip

Archive:  /content/AffordanceNet/full-shape.zip
  inflating: /content/AffordanceNet/full_shape_train_data.pkl  
  inflating: /content/AffordanceNet/full_shape_val_data.pkl  
Archive:  /content/AffordanceNet/partial.zip
  inflating: /content/AffordanceNet/partial_train_data.pkl  
  inflating: /content/AffordanceNet/partial_val_data.pkl  
Archive:  /content/AffordanceNet/rotate.zip
  inflating: /content/AffordanceNet/rotate_val_data.pkl  


In [ ]:
!pip install torch torchvision tqdm numpy scikit-learn

In [ ]:
# ---------- Cell 2: CONFIGURATION – CHANGE HERE FOR EACH ABLATION ----------
import os

# ┌─────────────────────────────────────────────────────────────┐
# │  ABLATION FLAGS (change ONLY these for each experiment)     │
# └─────────────────────────────────────────────────────────────┘
OCCLUSION_PROB = 0.5          # 0.0 = no occlusion augmentation
USE_BALANCED_SAMPLER = False   # False = random sampling
USE_FOCAL_LOSS = True         # False = plain BCEWithLogitsLoss

# Fixed settings (do not change)
NUM_EPOCHS = 50
BATCH_SIZE = 8
LEARNING_RATE = 5e-4
INPUT_DIM = 3                  # XYZ only
K_NEIGHBORS = 20

# Automatically generate experiment name and file paths
exp_parts = []
if OCCLUSION_PROB == 0.0:
    exp_parts.append("no_occ")
else:
    exp_parts.append("occ")
if not USE_BALANCED_SAMPLER:
    exp_parts.append("no_bal")
if not USE_FOCAL_LOSS:
    exp_parts.append("no_focal")
EXPERIMENT_NAME = "_".join(exp_parts) if exp_parts else "baseline"
SAVE_PATH = f"/content/output/best_abl_{EXPERIMENT_NAME}.pth"
HIST_PATH = f"/content/output/history_abl_{EXPERIMENT_NAME}.json"

print(f"{'='*60}")
print(f"  ABLATION: {EXPERIMENT_NAME}")
print(f"  Occlusion prob: {OCCLUSION_PROB}")
print(f"  Balanced sampler: {USE_BALANCED_SAMPLER}")
print(f"  Focal loss: {USE_FOCAL_LOSS}")
print(f"{'='*60}")

  ABLATION: occ_no_bal
  Occlusion prob: 0.5
  Balanced sampler: False
  Focal loss: True


In [ ]:
# ---------- Cell 3: Load data and configure split ----------
import pickle
import numpy as np
from collections import defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
from tqdm import tqdm

with open("/content/AffordanceNet/full_shape_train_data.pkl", "rb") as f:
    train_data = pickle.load(f)
with open("/content/AffordanceNet/full_shape_val_data.pkl", "rb") as f:
    val_data = pickle.load(f)

AFFORDANCE_NAMES = train_data[0]['affordance']

# Category -> affordances
cat_affs = defaultdict(set)
for s in train_data:
    cat = s['semantic class']
    for aff in AFFORDANCE_NAMES:
        if (s['full_shape']['label'][aff].flatten() > 0.5).any():
            cat_affs[cat].add(aff)

# Selected affordances (geometric)
SELECTED_CLASSES = ['grasp', 'contain', 'wrap_grasp', 'pourable']
SELECTED_INDICES = [AFFORDANCE_NAMES.index(c) for c in SELECTED_CLASSES]
NUM_CLASSES = len(SELECTED_CLASSES)

# Zero-shot split (same as before)
TEST_CATEGORIES = ['Mug', 'Vase', 'Scissors']
all_cats = set(s['semantic class'] for s in train_data)
TEST_CATEGORIES = [c for c in TEST_CATEGORIES if c in all_cats]

def has_positive(sample):
    for idx in SELECTED_INDICES:
        aff = AFFORDANCE_NAMES[idx]
        if (sample['full_shape']['label'][aff].flatten() > 0.5).any():
            return True
    return False

train_split = [s for s in train_data if s['semantic class'] not in TEST_CATEGORIES and has_positive(s)]
val_split   = [s for s in val_data   if s['semantic class'] not in TEST_CATEGORIES and has_positive(s)]
test_split  = [s for s in train_data if s['semantic class'] in TEST_CATEGORIES and has_positive(s)]

# Class weights for loss (only used if focal loss enabled)
total_pos = np.zeros(NUM_CLASSES)
total_all = np.zeros(NUM_CLASSES)
for s in train_split:
    label_dict = s['full_shape']['label']
    for i, idx in enumerate(SELECTED_INDICES):
        aff = AFFORDANCE_NAMES[idx]
        mask = label_dict[aff].flatten()
        total_pos[i] += (mask > 0.5).sum()
        total_all[i] += len(mask)
total_neg = total_all - total_pos
pos_weights = np.clip(total_neg / (total_pos + 1), 1.0, 50.0)

print(f"Train: {len(train_split)}, Val: {len(val_split)}, Test: {len(test_split)}")


Train: 3550, Val: 498, Test: 917


In [ ]:
# ---------- Cell 4: Precompute features (no geometric features, just XYZ) ----------
def precompute_xyz(data_list, desc=""):
    augmented = []
    for s in tqdm(data_list, desc=desc):
        pts = s['full_shape']['coordinate'].astype(np.float32)
        augmented.append({
            'features': pts,                     # (N,3)
            'labels': s['full_shape']['label'],
            'category': s['semantic class']
        })
    return augmented

train_aug = precompute_xyz(train_split, "Train XYZ")
val_aug   = precompute_xyz(val_split,   "Val XYZ")
test_aug  = precompute_xyz(test_split,  "Test XYZ")


Test XYZ: 100%|██████████| 917/917 [00:00<00:00, 49408.14it/s]


In [ ]:
# ---------- Cell 5: Oversample rare classes (same logic) ----------
class_sample_counts = np.zeros(NUM_CLASSES)
for s in train_aug:
    for i, idx in enumerate(SELECTED_INDICES):
        aff = AFFORDANCE_NAMES[idx]
        if (s['labels'][aff].flatten() > 0.5).any():
            class_sample_counts[i] += 1

max_count = class_sample_counts.max()
oversample_factors = {}
for i in range(NUM_CLASSES):
    factor = int(np.ceil(max_count / (class_sample_counts[i] + 1)))
    oversample_factors[i] = max(1, min(factor, 8))

oversampled_train = []
for s in train_aug:
    oversampled_train.append(s)
    best_factor = 1
    for i, idx in enumerate(SELECTED_INDICES):
        aff = AFFORDANCE_NAMES[idx]
        if (s['labels'][aff].flatten() > 0.5).any():
            best_factor = max(best_factor, oversample_factors[i])
    for _ in range(best_factor - 1):
        oversampled_train.append(s)

print(f"Oversampled: {len(train_aug)} -> {len(oversampled_train)}")


Oversampled: 3550 -> 9002


In [ ]:
# ---------- Cell 6: Dataset class ----------
class AffDataset(Dataset):
    def __init__(self, data, selected_indices, affordance_names,
                 num_points=2048, occlusion_prob=0.0, crop_angle=120):
        self.data = data
        self.selected_indices = selected_indices
        self.affordance_names = affordance_names
        self.num_points = num_points
        self.occlusion_prob = occlusion_prob
        self.crop_angle = crop_angle

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        pts = sample['features'].copy()    # (N,3)
        label_dict = sample['labels']

        masks = []
        for aff_idx in self.selected_indices:
            aff = self.affordance_names[aff_idx]
            masks.append((label_dict[aff].flatten() > 0.5).astype(np.float32))
        labels = np.stack(masks, axis=1)

        if np.random.random() < self.occlusion_prob:
            coords = pts[:, :3]
            centroid = coords.mean(axis=0)
            centered = coords - centroid
            norms = np.linalg.norm(centered, axis=1)
            norms[norms < 1e-8] = 1e-8
            dirs = centered / norms[:, None]
            theta = np.random.uniform(0, 2*np.pi)
            phi = np.random.uniform(0, np.pi)
            view = np.array([np.sin(phi)*np.cos(theta),
                             np.sin(phi)*np.sin(theta),
                             np.cos(phi)])
            visible = (dirs @ view) > np.cos(np.radians(self.crop_angle/2))
            if visible.sum() >= 64:
                pts = pts[visible]
                labels = labels[visible]

        # Normalise XYZ
        coords = pts[:, :3].copy()
        centroid = coords.mean(axis=0)
        coords -= centroid
        md = np.max(np.linalg.norm(coords, axis=1))
        if md > 1e-8: coords /= md
        pts[:, :3] = coords

        # Subsample
        N = len(pts)
        if N >= self.num_points:
            pos_mask = labels.sum(axis=1) > 0.5
            pos_idx = np.where(pos_mask)[0]
            neg_idx = np.where(~pos_mask)[0]
            if len(pos_idx) <= self.num_points:
                n_neg = self.num_points - len(pos_idx)
                neg_chosen = np.random.choice(neg_idx, min(n_neg, len(neg_idx)), replace=False) if len(neg_idx) > 0 else np.array([], dtype=int)
                chosen = np.concatenate([pos_idx, neg_chosen])
            else:
                chosen = np.random.choice(pos_idx, self.num_points, replace=False)
            if len(chosen) < self.num_points:
                extra = np.random.choice(N, self.num_points - len(chosen), replace=True)
                chosen = np.concatenate([chosen, extra])
        else:
            chosen = np.random.choice(N, self.num_points, replace=True)

        return (torch.tensor(pts[chosen], dtype=torch.float32),
                torch.tensor(labels[chosen], dtype=torch.float32))


In [ ]:
# ---------- Cell 7: Model (same DGCNNAblation as before) ----------
def knn(x, k=20):
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x ** 2, dim=1, keepdim=True)
    pw = -xx - inner - xx.transpose(2, 1)
    return pw.topk(k=k, dim=-1)[1]

def get_graph_feature(x, k=20, idx=None):
    B, C, N = x.shape
    if idx is None: idx = knn(x, k=k)
    device = x.device
    idx_base = torch.arange(0, B, device=device).view(-1, 1, 1) * N
    idx_flat = (idx + idx_base).view(-1)
    x_t = x.transpose(2, 1).contiguous().view(B * N, -1)
    feat = x_t[idx_flat].view(B, N, k, C)
    x_rep = x.transpose(2, 1).view(B, N, 1, C).expand(-1, -1, k, -1)
    return torch.cat([feat - x_rep, x_rep], dim=3).permute(0, 3, 1, 2).contiguous()

class DGCNNAblation(nn.Module):
    def __init__(self, num_classes, input_dim=3, k=20):
        super().__init__()
        self.k = k
        self.input_dim = input_dim
        self.conv1 = nn.Sequential(
            nn.Conv2d(input_dim*2, 64, 1, bias=False),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.conv2 = nn.Sequential(
            nn.Conv2d(128, 64, 1, bias=False),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 128, 1, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        self.head = nn.Sequential(
            nn.Conv1d(512, 256, 1, bias=False),
            nn.BatchNorm1d(256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Conv1d(256, 128, 1, bias=False),
            nn.BatchNorm1d(128), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Conv1d(128, num_classes, 1))

    def forward(self, x):
        B, C, N = x.shape
        xyz = x[:, :3, :]
        idx1 = knn(xyz, k=self.k)
        edge = get_graph_feature(x, k=self.k, idx=idx1)
        x1 = self.conv1(edge).max(dim=-1)[0]
        edge = get_graph_feature(x1, k=self.k)
        x2 = self.conv2(edge).max(dim=-1)[0]
        edge = get_graph_feature(x2, k=self.k)
        x3 = self.conv3(edge).max(dim=-1)[0]
        x_local = torch.cat([x1, x2, x3], dim=1)
        x_global = x_local.max(dim=-1, keepdim=True)[0].expand(-1, -1, N)
        out = self.head(torch.cat([x_local, x_global], dim=1))
        return out.permute(0, 2, 1)


In [ ]:
# ---------- Cell 8: Loss function (focal or BCE) ----------
class FocalBCEDiceLoss(nn.Module):
    def __init__(self, pos_weight, gamma=2.0, smooth=1.0):
        super().__init__()
        self.register_buffer('pos_weight', pos_weight)
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, pred, target):
        bce_raw = nn.functional.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pw = self.pos_weight.unsqueeze(0).unsqueeze(0)
        weight = torch.where(target > 0.5, pw, torch.ones_like(pw))
        pt = torch.sigmoid(pred)
        pt = torch.where(target > 0.5, pt, 1 - pt)
        focal = (1 - pt) ** self.gamma
        bce = (focal * weight * bce_raw).mean()

        pred_sig = torch.sigmoid(pred)
        dice = 0.0
        for c in range(pred.size(2)):
            inter = (pred_sig[..., c] * target[..., c]).sum()
            union = pred_sig[..., c].sum() + target[..., c].sum()
            dice += 1 - (2.*inter + self.smooth) / (union + self.smooth)
        return bce + dice / pred.size(2)


In [ ]:
# ---------- Cell 9: Balanced sampler (if enabled) ----------
class BalancedSampler(Sampler):
    def __init__(self, dataset, batch_size, num_classes, affordance_names, selected_indices):
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.class_to_idx = defaultdict(list)
        for i, s in enumerate(dataset.data):
            for c_idx, aff_idx in enumerate(selected_indices):
                aff = affordance_names[aff_idx]
                if (s['labels'][aff].flatten() > 0.5).any():
                    self.class_to_idx[c_idx].append(i)
        self.num_samples = len(dataset)
        self.num_batches = self.num_samples // batch_size

    def __iter__(self):
        pools = {c: np.random.permutation(v).tolist() for c, v in self.class_to_idx.items()}
        batches = []
        for _ in range(self.num_batches):
            batch, used = [], set()
            for c in range(self.num_classes):
                if not pools[c]:
                    pools[c] = np.random.permutation(self.class_to_idx[c]).tolist()
                idx = pools[c].pop()
                if idx not in used:
                    batch.append(idx)
                    used.add(idx)
            filler = np.random.permutation(self.num_samples)
            for idx in filler:
                if len(batch) >= self.batch_size: break
                if int(idx) not in used:
                    batch.append(int(idx))
                    used.add(int(idx))
            while len(batch) < self.batch_size:
                batch.append(np.random.randint(self.num_samples))
            batches.append(batch)
        np.random.shuffle(batches)
        for b in batches: yield b

    def __len__(self): return self.num_batches


In [ ]:
# ---------- Cell 10: Create datasets and loaders ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_ds = AffDataset(oversampled_train, SELECTED_INDICES, AFFORDANCE_NAMES,
                      num_points=2048, occlusion_prob=OCCLUSION_PROB, crop_angle=120)
val_ds   = AffDataset(val_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
                      num_points=2048, occlusion_prob=0.0)
test_clean_ds = AffDataset(test_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
                           num_points=2048, occlusion_prob=0.0)
test_occ_ds   = AffDataset(test_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
                           num_points=2048, occlusion_prob=1.0, crop_angle=120)

if USE_BALANCED_SAMPLER:
    sampler = BalancedSampler(train_ds, BATCH_SIZE, NUM_CLASSES, AFFORDANCE_NAMES, SELECTED_INDICES)
    train_loader = DataLoader(train_ds, batch_sampler=sampler, num_workers=2)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_clean_loader = DataLoader(test_clean_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_occ_loader   = DataLoader(test_occ_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


In [14]:
# ---------- Cell 11: Training ----------
model = DGCNNAblation(num_classes=NUM_CLASSES, input_dim=INPUT_DIM, k=K_NEIGHBORS).to(device)
with torch.no_grad():
    model.head[-1].bias.data.fill_(1.0)

if USE_FOCAL_LOSS:
    pw_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(device)
    criterion = FocalBCEDiceLoss(pw_tensor, gamma=2.0).to(device)
else:
    # Plain BCE + Dice (no focal, no pos_weight)
    def dice_loss(pred, target):
        pred_sig = torch.sigmoid(pred)
        dice = 0.0
        for c in range(pred.size(2)):
            inter = (pred_sig[..., c] * target[..., c]).sum()
            union = pred_sig[..., c].sum() + target[..., c].sum()
            dice += 1 - (2.*inter + 1.0) / (union + 1.0)
        return dice / pred.size(2)
    def bce_dice(pred, target):
        bce = nn.functional.binary_cross_entropy_with_logits(pred, target)
        return bce + dice_loss(pred, target)
    criterion = bce_dice

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)

os.makedirs("/content/output", exist_ok=True)
best_val = 0.0
history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for pts, labels in tqdm(train_loader, desc=f"[{EXPERIMENT_NAME}] Epoch {epoch+1}/{NUM_EPOCHS}"):
        pts = pts.permute(0,2,1).to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(pts), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)

    # Validation (seen clean)
    model.eval()
    iou_c = torch.zeros(NUM_CLASSES, device=device)
    cnt_c = torch.zeros(NUM_CLASSES, device=device)
    with torch.no_grad():
        for pts, labels in val_loader:
            pts = pts.permute(0,2,1).to(device)
            labels = labels.to(device)
            pred = (torch.sigmoid(model(pts)) > 0.5).float()
            for b in range(pts.size(0)):
                for c in range(NUM_CLASSES):
                    gt = labels[b,:,c]
                    if gt.sum() > 0:
                        inter = (pred[b,:,c] * gt).sum()
                        union = (pred[b,:,c] + gt).clamp(0,1).sum()
                        iou_c[c] += inter / (union + 1e-8)
                        cnt_c[c] += 1

    avg_iou = iou_c / (cnt_c + 1e-8)
    miou = avg_iou.mean().item()
    history.append({'epoch': epoch+1, 'loss': avg_loss, 'miou': miou,
                    'per_class': avg_iou.cpu().tolist()})

    cls = ', '.join(f"{SELECTED_CLASSES[i]}={avg_iou[i].item():.3f}" for i in range(NUM_CLASSES))
    print(f"  Loss {avg_loss:.4f} | Val mIoU {miou:.4f} | {cls}")

    scheduler.step()
    if miou > best_val:
        best_val = miou
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  ✅ Saved (best={best_val:.4f})")

print(f"\n🏆 [{EXPERIMENT_NAME}] Best Val mIoU: {best_val:.4f}")


[occ_no_bal] Epoch 1/50: 100%|██████████| 1126/1126 [02:39<00:00,  7.08it/s]


  Loss 1.2393 | Val mIoU 0.1559 | grasp=0.104, contain=0.279, wrap_grasp=0.082, pourable=0.158
  ✅ Saved (best=0.1559)


[occ_no_bal] Epoch 2/50: 100%|██████████| 1126/1126 [02:42<00:00,  6.94it/s]


  Loss 1.1995 | Val mIoU 0.1570 | grasp=0.107, contain=0.278, wrap_grasp=0.081, pourable=0.162
  ✅ Saved (best=0.1570)


[occ_no_bal] Epoch 3/50: 100%|██████████| 1126/1126 [02:41<00:00,  6.95it/s]


...

  Loss 1.1298 | Val mIoU 0.2075 | grasp=0.142, contain=0.311, wrap_grasp=0.084, pourable=0.294


[occ_no_bal] Epoch 49/50: 100%|██████████| 1126/1126 [02:42<00:00,  6.94it/s]


  Loss 1.1290 | Val mIoU 0.1963 | grasp=0.136, contain=0.279, wrap_grasp=0.082, pourable=0.288


[occ_no_bal] Epoch 50/50: 100%|██████████| 1126/1126 [02:42<00:00,  6.94it/s]


  Loss 1.1329 | Val mIoU 0.2001 | grasp=0.134, contain=0.301, wrap_grasp=0.083, pourable=0.283

🏆 [occ_no_bal] Best Val mIoU: 0.2116


In [15]:
# ---------- Cell 12: Four‑condition evaluation ----------
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.eval()

def evaluate(loader, desc):
    iou_c = np.zeros(NUM_CLASSES)
    cnt_c = np.zeros(NUM_CLASSES)
    with torch.no_grad():
        for pts, labels in tqdm(loader, desc=desc):
            pts = pts.permute(0,2,1).to(device)
            pred = (torch.sigmoid(model(pts)) > 0.5).cpu().numpy()
            labels = labels.numpy()
            for b in range(pred.shape[0]):
                for c in range(NUM_CLASSES):
                    gt = labels[b,:,c]
                    if gt.sum() > 0:
                        inter = (pred[b,:,c] * gt).sum()
                        union = np.clip(pred[b,:,c] + gt, 0, 1).sum()
                        iou_c[c] += inter / (union + 1e-8)
                        cnt_c[c] += 1
    avg = iou_c / (cnt_c + 1e-8)
    m = avg.mean()
    print(f"\n{desc} — mIoU: {m:.4f}")
    for i, n in enumerate(SELECTED_CLASSES):
        print(f"  {n}: {avg[i]:.4f} (n={int(cnt_c[i])})")
    return m, avg

seen_clean_ds = AffDataset(val_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
                           num_points=2048, occlusion_prob=0.0)
seen_occ_ds   = AffDataset(val_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
                           num_points=2048, occlusion_prob=1.0, crop_angle=120)
seen_clean_ldr = DataLoader(seen_clean_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
seen_occ_ldr   = DataLoader(seen_occ_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\n{'='*60}")
print(f"  RESULTS: {EXPERIMENT_NAME.upper()} (OCC={OCCLUSION_PROB}, BAL={USE_BALANCED_SAMPLER}, FOCAL={USE_FOCAL_LOSS})")
print(f"{'='*60}")

r1, _ = evaluate(seen_clean_ldr,    "Seen / Clean")
r2, _ = evaluate(seen_occ_ldr,      "Seen / Occluded")
r3, _ = evaluate(test_clean_loader, "Unseen / Clean")
r4, _ = evaluate(test_occ_loader,   "Unseen / Occluded")

print(f"\n{'='*50}")
print(f"  SUMMARY — {EXPERIMENT_NAME.upper()}")
print(f"{'='*50}")
print(f"{'':20s} | {'Clean':>8s} | {'Occluded':>8s}")
print(f"{'-'*20}-+-{'-'*8}-+-{'-'*8}")
print(f"{'Seen':20s} | {r1:8.4f} | {r2:8.4f}")
print(f"{'Unseen (zero-shot)':20s} | {r3:8.4f} | {r4:8.4f}")
print(f"{'='*50}")

# Save history
import json
with open(HIST_PATH, 'w') as f:
    json.dump(history, f)
print(f"History saved to {HIST_PATH}")



  RESULTS: OCC_NO_BAL (OCC=0.5, BAL=False, FOCAL=True)


Seen / Clean: 100%|██████████| 63/63 [00:06<00:00, 10.44it/s]



Seen / Clean — mIoU: 0.2116
  grasp: 0.1488 (n=177)
  contain: 0.3183 (n=361)
  wrap_grasp: 0.0833 (n=57)
  pourable: 0.2961 (n=82)


Seen / Occluded: 100%|██████████| 63/63 [00:05<00:00, 10.57it/s]



Seen / Occluded — mIoU: 0.1711
  grasp: 0.1413 (n=105)
  contain: 0.2582 (n=256)
  wrap_grasp: 0.0909 (n=49)
  pourable: 0.1941 (n=49)


Unseen / Clean: 100%|██████████| 115/115 [00:10<00:00, 10.70it/s]



Unseen / Clean — mIoU: 0.0962
  grasp: 0.0170 (n=176)
  contain: 0.1621 (n=817)
  wrap_grasp: 0.0290 (n=861)
  pourable: 0.1768 (n=811)


Unseen / Occluded: 100%|██████████| 115/115 [00:10<00:00, 10.76it/s]


Unseen / Occluded — mIoU: 0.0991
  grasp: 0.0681 (n=94)
  contain: 0.1407 (n=490)
  wrap_grasp: 0.0517 (n=634)
  pourable: 0.1359 (n=471)

  SUMMARY — OCC_NO_BAL
                     |    Clean | Occluded
---------------------+----------+---------
Seen                 |   0.2116 |   0.1711
Unseen (zero-shot)   |   0.0962 |   0.0991
History saved to /content/output/history_abl_occ_no_bal.json
